# Agent AI Bootcamp

Code from the section notes goes into cells below. Keys are read from `.env` in the project root.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

# Groq (OpenAI-compatible). To use OpenAI instead, replace with: openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

Quick API check. Run this after putting your real Groq key in `.env`; a 401 error means the key is wrong or still the placeholder.

In [2]:
response = openai_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Reply with the single word: ok"}],
)
print(response.choices[0].message.content)

ok


## 03 RAG: Retrieval-Augmented Generation

Course notes use OpenAI `responses.create` with `gpt-5.4-mini`. We use Groq, so the same idea is written with `chat.completions` and `openai/gpt-oss-120b`.

In [3]:
MODEL = "openai/gpt-oss-120b"

def llm(prompt):
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

In [4]:
llm("Hey, what's up?")

"Hey there! I'm doing great—ready to help with whatever you need. What's on your mind today?"

### Plain LLM lacks our data
A course-specific question gets a generic answer: the model knows nothing about Zoomcamp enrollment.

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely—you can enroll at any time! 🎉 Most of our courses are open‑enrollment, which means you can start whenever you’re ready. Here’s a quick rundown of what to do next:

1. **Visit the Course Page**  
   Go to the course’s landing page (or the dashboard if you’re already logged in). You should see a “Enroll Now” or “Join Course” button.

2. **Create or Log In to Your Account**  
   - If you don’t have an account yet, click **Sign Up** and fill in your name, email, and a password.  
   - If you already have an account, just log in with your credentials.

3. **Complete the Enrollment Form**  
   Some courses ask a few quick questions (e.g., your skill level or learning goals) to personalize the experience. Fill those out and click **Submit**.

4. **Confirm Your Payment (if applicable)**  
   - **Free courses:** No payment required—just confirm your enrollment.  
   - **Paid courses:** You’ll be directed to a secure checkout page where you can pay via credit/debit card, PayPal, or ot

### Adding context manually
Paste FAQ entries into the prompt. This is RAG done by hand.

In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes—you can still join the course. If you’d like to earn a certificate, just be sure to submit your project before the submission deadline.


### RAG = search + prompt + LLM

```
User question -> search(knowledge base) -> top documents -> build_prompt(question, docs) -> llm(prompt) -> answer
```

The LLM only sees the documents we hand it, so answers are grounded in our data. Retrieval quality decides answer quality. `search` and `build_prompt` are implemented in the next sections (dataset + minsearch); the skeleton below is the whole architecture.

In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

# search() and build_prompt() come in the next section